# Bias Mitigation Pipeline: Learning Fair Representations (LFR)
Learning Fair Representations (LFR) — Zemel et al., ICML 2013 (pre-processing)
**Datasets:** Adult Census Income, Statlog German Credit, COMPAS (ProPublica two-year recidivism)

All three datasets are read from **local folders**:

```
./adult/adult.csv
./statlog+german+credit+data/german.data
./compass/propublicaCompassRecividism_data_fairml/propublica_data_for_fairml.csv
```

## 1. Environment setup & imports

In [5]:
# Uncomment to install the dependencies if needed
# import sys
# !{sys.executable} -m pip install aif360 scikit-learn pandas numpy scipy matplotlib

In [6]:
import os
import time
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger().setLevel(logging.ERROR)   # silence AIF360 optional-dependency warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from aif360.datasets import StandardDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import LFR
import inspect
import scipy.optimize as _optim
import aif360.algorithms.preprocessing.lfr as _lfr_module
if "disp" not in inspect.signature(_optim.fmin_l_bfgs_b).parameters:
    class _OptimCompat:
        def __getattr__(self, name):
            return getattr(_optim, name)
        @staticmethod
        def fmin_l_bfgs_b(*args, disp=None, **kwargs):
            return _optim.fmin_l_bfgs_b(*args, **kwargs)
    _lfr_module.optim = _OptimCompat()
    print("Applied SciPy/AIF360 LFR compatibility patch.")


SEED = 42
np.random.seed(SEED)

# Folder that contains the dataset sub-folders (by default: the notebook's folder)
DATA_DIR = "."
PATHS = {
    "Adult":  "adult/adult.csv",
    "German": "statlog+german+credit+data/german.data",
    "COMPAS": "compass/propublicaCompassRecividism_data_fairml/propublica_data_for_fairml.csv",
}


Applied SciPy/AIF360 LFR compatibility patch.


## 2. Dataset loading & configuration
| Dataset | Protected attribute | Privileged (1) | Unprivileged (0) | Favorable label |
|---|---|---|---|---|
| Adult | `sex` | Male | Female | income > 50K |
| German | `age` | age > 25 | age ≤ 25 | good credit risk |
| COMPAS | `race` | Caucasian | African-American | **no** two-year recidivism |

Notes on the raw files:
* **Adult** (`adult.csv`): missing values are coded as `?` and those rows are dropped. `fnlwgt` (census sampling weight) is removed; `education` is represented only by its ordinal twin `education.num`, and `native.country` is reduced to *US / non-US*. This keeps Adult at 46 features instead of ~100, which matters because AIF360's LFR has `k·(d+1)` parameters optimized with finite-difference gradients (with ~100 features the optimizer only manages a handful of iterations and LFR collapses to a constant predictor).
* **German** (`german.data`): space-separated, no header, symbolic codes (`A11`, …); target `1 = good`, `2 = bad`. Column names follow `german.doc`. `personal_status` mixes sex and marital status, so it is replaced by a binary `sex` feature.
* **COMPAS**: `compas-scores-raw.csv` contains only COMPAS assessments and **no recidivism outcome**, so it cannot be used for this task. `propublica_data_for_fairml.csv` is the ProPublica two-year-recidivism cohort already cleaned with ProPublica's filters (6,172 people), with the `Two_yr_Recidivism` label. Race is one-hot encoded; Caucasian is the reference category (all race dummies = 0). We keep the African-American and Caucasian defendants (the comparison used by ProPublica).

All categorical features are one-hot encoded by `StandardDataset`.

In [7]:
def load_adult(path):
    df = pd.read_csv(path, na_values="?", skipinitialspace=True)
    df = df.dropna().reset_index(drop=True)
    df["income"] = df["income"].str.strip().isin([">50K", ">50K."]).astype(float)
    df["sex"] = (df["sex"].str.strip() == "Male").astype(float)
    # Compact encoding (keeps LFR's parameter count manageable, see Section 4):
    #  - `education.num` is the ordinal version of `education` -> keep only the ordinal one
    #  - `native.country` (41 levels, ~91% United-States) -> binary native_us
    df["native_us"] = (df["native.country"] == "United-States").astype(float)
    return StandardDataset(
        df=df,
        label_name="income",
        favorable_classes=[1.0],
        protected_attribute_names=["sex"],
        privileged_classes=[[1.0]],
        categorical_features=["workclass", "marital.status", "occupation", "relationship", "race"],
        features_to_drop=["fnlwgt", "education", "native.country"],
    )


GERMAN_COLUMNS = [
    "status", "month", "credit_history", "purpose", "credit_amount", "savings",
    "employment", "investment_as_income_percentage", "personal_status", "other_debtors",
    "residence_since", "property", "age", "installment_plans", "housing",
    "number_of_credits", "skill_level", "people_liable_for", "telephone",
    "foreign_worker", "credit",
]

def load_german(path):
    df = pd.read_csv(path, sep=r"\s+", header=None, names=GERMAN_COLUMNS)
    # A91, A93, A94 = male ; A92, A95 = female (see german.doc)
    df["sex"] = df["personal_status"].map({"A91": 1.0, "A93": 1.0, "A94": 1.0,
                                           "A92": 0.0, "A95": 0.0})
    df["credit"] = (df["credit"] == 1).astype(float)          # 1 = good, 2 = bad
    return StandardDataset(
        df=df,
        label_name="credit",
        favorable_classes=[1.0],
        protected_attribute_names=["age"],
        privileged_classes=[lambda x: x > 25],                # age > 25 -> 1, else 0
        categorical_features=["status", "credit_history", "purpose", "savings", "employment",
                              "other_debtors", "property", "installment_plans", "housing",
                              "skill_level", "telephone", "foreign_worker"],
        features_to_drop=["personal_status"],
    )


def load_compas(path):
    df = pd.read_csv(path)
    other_races = ["Asian", "Hispanic", "Native_American", "Other"]
    is_caucasian = df[["African_American"] + other_races].sum(axis=1) == 0
    df = df[(df["African_American"] == 1) | is_caucasian].copy()
    df["race"] = (df["African_American"] == 0).astype(float)  # Caucasian = 1, African-American = 0
    df = df.drop(columns=["African_American"] + other_races).reset_index(drop=True)
    return StandardDataset(
        df=df,
        label_name="Two_yr_Recidivism",
        favorable_classes=[0],                               
        protected_attribute_names=["race"],
        privileged_classes=[[1.0]],
        categorical_features=[],
    )


def load_datasets():
    # Default LFR settings (used directly if TUNE_LFR = False). Values chosen with the
    # validation procedure of Section 4.
    return {
        "Adult":  {"data": load_adult(PATHS["Adult"]),   "protected_attr": "sex",
                   "privileged": [{"sex": 1.0}],  "unprivileged": [{"sex": 0.0}],
                   "lfr_params": {"k": 5, "Ax": 0.01, "Ay": 1.0, "Az": 2.0, "maxiter": 5000, "maxfun": 20000},
                   "az_grid": [2.0, 5.0, 10.0]},
        "German": {"data": load_german(PATHS["German"]), "protected_attr": "age",
                   "privileged": [{"age": 1.0}],  "unprivileged": [{"age": 0.0}],
                   "lfr_params": {"k": 10, "Ax": 0.01, "Ay": 1.0, "Az": 2.0, "maxiter": 5000, "maxfun": 20000},
                   "az_grid": [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 25.0]},
        "COMPAS": {"data": load_compas(PATHS["COMPAS"]), "protected_attr": "race",
                   "privileged": [{"race": 1.0}], "unprivileged": [{"race": 0.0}],
                   "lfr_params": {"k": 10, "Ax": 0.01, "Ay": 1.0, "Az": 1.0, "maxiter": 5000, "maxfun": 20000},
                   "az_grid": [0.5, 1.0, 2.0, 5.0, 10.0, 25.0, 50.0]},
    }


datasets = load_datasets()
for name, cfg in datasets.items():
    ds = cfg["data"]
    prot = ds.protected_attributes[:, 0]
    print(f"{name:7s}: {ds.features.shape[0]:6d} samples, {ds.features.shape[1]:3d} features | "
          f"privileged share = {prot.mean():.3f} | favorable-label rate = "
          f"{(ds.labels.ravel() == ds.favorable_label).mean():.3f}")

Adult  :  30162 samples,  46 features | privileged share = 0.676 | favorable-label rate = 0.249
German :   1000 samples,  58 features | privileged share = 0.810 | favorable-label rate = 0.700
COMPAS :   5278 samples,   7 features | privileged share = 0.398 | favorable-label rate = 0.530


## 3. Fairness & predictive evaluation functions
* **DP_Diff** (demographic / statistical parity difference)
* **DIR** (disparate impact ratio) 
* **Consistency** 

Consistency depends on Euclidean distances, so the kNN search is always done on **standardized original features** 

In [8]:
def fairness_metrics(dataset, unprivileged_groups, privileged_groups, n_neighbors=5):
    """DP_Diff, DIR and Consistency of the labels stored in an AIF360 dataset."""
    metric = BinaryLabelDatasetMetric(dataset, unprivileged_groups=unprivileged_groups,
                                      privileged_groups=privileged_groups)
    dp_diff = abs(metric.statistical_parity_difference())
    dir_val = metric.disparate_impact()
    consistency = float(np.ravel(metric.consistency(n_neighbors=n_neighbors))[0])
    return dp_diff, dir_val, consistency


def standardized_copy(dataset):
    """Copy of a dataset with z-scored features (only used for the kNN in Consistency)."""
    out = dataset.copy(deepcopy=True)
    out.features = StandardScaler().fit_transform(dataset.features)
    return out


def stratified_split_aif360(dataset, test_size=0.3, seed=SEED):
    """70/30 split stratified on the joint (protected attribute, label) groups."""
    prot = dataset.protected_attributes[:, 0].astype(int)
    labels = dataset.labels.ravel().astype(int)
    strata = np.array([f"{p}_{y}" for p, y in zip(prot, labels)])
    idx = np.arange(len(labels))
    train_idx, test_idx = train_test_split(idx, test_size=test_size,
                                           random_state=seed, stratify=strata)
    return dataset.subset(train_idx), dataset.subset(test_idx)


def scale_datasets(train_ds, test_ds):
    """Standardize features with statistics from the TRAINING set only (no leakage)."""
    scaler = StandardScaler().fit(train_ds.features)
    train_s, test_s = train_ds.copy(deepcopy=True), test_ds.copy(deepcopy=True)
    train_s.features = scaler.transform(train_ds.features)
    test_s.features = scaler.transform(test_ds.features)
    return train_s, test_s


def evaluate_predictions(test_ds, y_pred, unprivileged_groups, privileged_groups):
    """Fairness of the predictions (kNN on the original standardized test features)
    plus predictive performance against the true test labels."""
    pred_ds = test_ds.copy(deepcopy=True)
    pred_ds.labels = np.asarray(y_pred, dtype=float).reshape(-1, 1)
    dp_diff, dir_val, consistency = fairness_metrics(pred_ds, unprivileged_groups, privileged_groups)

    y_true = test_ds.labels.ravel()
    fav = test_ds.favorable_label
    return {
        "DP_Diff": dp_diff,
        "DIR": dir_val,
        "Consistency": consistency,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Acc": balanced_accuracy_score(y_true, y_pred),
        "F1_Score": f1_score(y_true, y_pred, pos_label=fav),
        "Pos_Rate": float(np.mean(np.asarray(y_pred) == fav)),
    }

## 4. Choosing the LFR fairness weight $A_z$ (validation split, no test data)
$A_z$ controls the fairness/utility trade-off. A value that is too large makes LFR map everybody to the same prototype mixture, so the downstream classifier predicts one class for everyone: DP_Diff = 0 and DIR = 1 but balanced accuracy = 0.5 (useless "fairness").  
To pick $A_z$ without touching the test set, the **training split** is divided again (75 % fit / 25 % validation, stratified the same way). For each $A_z$ in a small grid we fit LFR + LR on the fit part and evaluate on the validation part. Selection rule: among settings whose validation DIR lies in the 80 %-rule band $[0.8, 1.25]$, take the one with the highest balanced accuracy; if none qualifies, take the one with DIR closest to 1.  


In [9]:
TUNE_LFR = True

def fit_lfr_and_predict(train_s, eval_s, unpriv, priv, p):
    lfr = LFR(unprivileged_groups=unpriv, privileged_groups=priv,
              k=p["k"], Ax=p["Ax"], Ay=p["Ay"], Az=p["Az"], seed=SEED, verbose=0)
    lfr.fit(train_s, maxiter=p["maxiter"], maxfun=p["maxfun"])
    clf = LogisticRegression(max_iter=2000, random_state=SEED)
    clf.fit(lfr.transform(train_s).features, train_s.labels.ravel())
    return clf.predict(lfr.transform(eval_s).features)


def tune_az(cfg):
    train_ds, _ = stratified_split_aif360(cfg["data"], test_size=0.3, seed=SEED)  # same split as Section 4
    fit_ds, val_ds = stratified_split_aif360(train_ds, test_size=0.25, seed=SEED)
    fit_s, val_s = scale_datasets(fit_ds, val_ds)
    rows = []
    for az in cfg["az_grid"]:
        p = dict(cfg["lfr_params"], Az=az)
        y_val = fit_lfr_and_predict(fit_s, val_s, cfg["unprivileged"], cfg["privileged"], p)
        rows.append({"Az": az, **evaluate_predictions(val_s, y_val, cfg["unprivileged"], cfg["privileged"])})
    res = pd.DataFrame(rows)
    ok = res[(res.DIR >= 0.8) & (res.DIR <= 1.25)]
    best = ok.loc[ok.Balanced_Acc.idxmax()] if len(ok) else res.loc[(res.DIR - 1).abs().idxmin()]
    return float(best.Az), res


if TUNE_LFR:
    for name, cfg in datasets.items():
        t0 = time.time()
        best_az, res = tune_az(cfg)
        cfg["lfr_params"]["Az"] = best_az
        print(f"{name}: selected Az = {best_az}  ({time.time() - t0:.0f}s)")
        display(res.round(4))
else:
    print("Using default LFR parameters:", {n: c["lfr_params"]["Az"] for n, c in datasets.items()})

Adult: selected Az = 2.0  (634s)


,Az,DP_Diff,DIR,Consistency,Accuracy,Balanced_Acc,F1_Score,Pos_Rate
0,2.0,0.0062,0.9679,0.9581,0.8197,0.7204,0.5907,0.1917
1,5.0,0.0374,0.8143,0.9592,0.8204,0.7199,0.5903,0.1894
2,10.0,0.0046,0.9409,0.9661,0.7615,0.5655,0.2676,0.0767


German: selected Az = 2.0  (72s)


,Az,DP_Diff,DIR,Consistency,Accuracy,Balanced_Acc,F1_Score,Pos_Rate
0,0.1,0.3039,0.6147,0.7840,0.7600,0.6998,0.8320,0.7314
1,0.5,0.1989,0.7433,0.7863,0.7429,0.6768,0.8207,0.7371
2,1.0,0.0100,0.9869,0.8126,0.7486,0.6703,0.8281,0.7657
3,2.0,0.0312,0.9605,0.8137,0.7657,0.6826,0.8417,0.7829
4,5.0,0.1112,1.1448,0.8491,0.7371,0.6461,0.8231,0.7886
5,10.0,0.0070,1.0071,0.9909,0.7029,0.5094,0.8243,0.9943
6,25.0,0.0070,1.0071,0.9909,0.7029,0.5094,0.8243,0.9943


COMPAS: selected Az = 1.0  (95s)


,Az,DP_Diff,DIR,Consistency,Accuracy,Balanced_Acc,F1_Score,Pos_Rate
0,0.5,0.1721,0.7446,0.9656,0.6948,0.6913,0.7224,0.5703
1,1.0,0.0557,0.9097,0.9647,0.6970,0.6928,0.7276,0.5833
2,2.0,0.0801,0.8730,0.9688,0.6851,0.6809,0.7167,0.5823
3,5.0,0.0675,0.8930,0.9786,0.6905,0.6859,0.7234,0.5898
4,10.0,0.0600,0.9160,0.9870,0.6732,0.6633,0.7294,0.6786
5,25.0,0.0385,0.9469,0.9846,0.6494,0.6380,0.7153,0.7024
6,50.0,0.0512,0.9305,0.9851,0.6483,0.6367,0.7152,0.7056


## 5. Full bias-mitigation pipeline on the three datasets
For every dataset we record:
1. **Original labels (full dataset, pre-split)** – assignment step 2.
2. **Original labels (test split)** – the ground truth of the held-out data, as a reference for the test-set results.
3. **Baseline LR (unmitigated)** – `LogisticRegression` on the standardized original features.
4. **LFR + LR (mitigated)** – LFR fit on the standardized training split only; both splits are mapped to the learned representation $\hat X$ and a `LogisticRegression` is trained on the transformed training data and evaluated on the transformed test data. The true training labels are used to fit the downstream classifier (LFR only changes the features).

AIF360's LFR optimizes $A_x L_x + A_y L_y + A_z L_z$ with L-BFGS-B and finite-difference gradients; `maxfun` is raised to 20 000 so the optimizer gets enough iterations.

In [10]:
def run_experiment(datasets):
    rows = []
    for name, cfg in datasets.items():
        print(f"{'=' * 72}\nDataset: {name}\n{'=' * 72}")
        ds, priv, unpriv = cfg["data"], cfg["privileged"], cfg["unprivileged"]

        # fairness of the ORIGINAL labels, before split / mitigation / training
        dp, di, cons = fairness_metrics(standardized_copy(ds), unpriv, priv)
        rows.append({"Dataset": name, "Stage": "1. Original labels (full, pre-split)",
                     "DP_Diff": dp, "DIR": di, "Consistency": cons})
        print(f"[Original labels ] DP_Diff={dp:.4f} | DIR={di:.4f} | Consistency={cons:.4f}")

        # stratified 70/30 split + scaling fit on train only
        train_ds, test_ds = stratified_split_aif360(ds, test_size=0.3, seed=SEED)
        train_s, test_s = scale_datasets(train_ds, test_ds)
        print(f"Split: train={train_s.features.shape[0]}, test={test_s.features.shape[0]}")

        dp, di, cons = fairness_metrics(test_s, unpriv, priv)
        rows.append({"Dataset": name, "Stage": "2. Original labels (test split)",
                     "DP_Diff": dp, "DIR": di, "Consistency": cons})

        # Unmitigated baseline
        base_clf = LogisticRegression(max_iter=2000, random_state=SEED)
        base_clf.fit(train_s.features, train_s.labels.ravel())
        m = evaluate_predictions(test_s, base_clf.predict(test_s.features), unpriv, priv)
        rows.append({"Dataset": name, "Stage": "3. Baseline LR (unmitigated)", **m})
        print(f"[Baseline LR     ] DP_Diff={m['DP_Diff']:.4f} | DIR={m['DIR']:.4f} | "
              f"Consistency={m['Consistency']:.4f} | Acc={m['Accuracy']:.4f} | BalAcc={m['Balanced_Acc']:.4f}")

        # LFR pre-processing, fit ONLY on the training data
        p = cfg["lfr_params"]
        print(f"LFR params: {p}")
        t0 = time.time()
        lfr = LFR(unprivileged_groups=unpriv, privileged_groups=priv,
                  k=p["k"], Ax=p["Ax"], Ay=p["Ay"], Az=p["Az"], seed=SEED, verbose=0)
        lfr.fit(train_s, maxiter=p["maxiter"], maxfun=p["maxfun"])
        train_t = lfr.transform(train_s)
        test_t = lfr.transform(test_s)
        print(f"LFR fitted in {time.time() - t0:.1f}s")

        # Downstream classifier on the fair representation (true training labels)
        fair_clf = LogisticRegression(max_iter=2000, random_state=SEED)
        fair_clf.fit(train_t.features, train_s.labels.ravel())
        y_pred_fair = fair_clf.predict(test_t.features)

        # evaluate final predictions on held-out test data
        m = evaluate_predictions(test_s, y_pred_fair, unpriv, priv)
        rows.append({"Dataset": name, "Stage": "4. LFR + LR (mitigated)", **m})
        print(f"[LFR + LR        ] DP_Diff={m['DP_Diff']:.4f} | DIR={m['DIR']:.4f} | "
              f"Consistency={m['Consistency']:.4f} | Acc={m['Accuracy']:.4f} | BalAcc={m['Balanced_Acc']:.4f}\n")
    return pd.DataFrame(rows)


results_df = run_experiment(datasets)

Dataset: Adult
[Original labels ] DP_Diff=0.2002 | DIR=0.3622 | Consistency=0.8327
Split: train=21113, test=9049
[Baseline LR     ] DP_Diff=0.2006 | DIR=0.2756 | Consistency=0.9478 | Acc=0.8448 | BalAcc=0.7676
LFR params: {'k': 5, 'Ax': 0.01, 'Ay': 1.0, 'Az': 2.0, 'maxiter': 5000, 'maxfun': 20000}
LFR fitted in 294.2s
[LFR + LR        ] DP_Diff=0.0283 | DIR=0.8672 | Consistency=0.9598 | Acc=0.8224 | BalAcc=0.7322

Dataset: German
[Original labels ] DP_Diff=0.1494 | DIR=0.7948 | Consistency=0.7402
Split: train=700, test=300
[Baseline LR     ] DP_Diff=0.1501 | DIR=0.8238 | Consistency=0.8573 | Acc=0.7700 | BalAcc=0.6675
LFR params: {'k': 10, 'Ax': 0.01, 'Ay': 1.0, 'Az': 2.0, 'maxiter': 5000, 'maxfun': 20000}
LFR fitted in 12.3s
[LFR + LR        ] DP_Diff=0.0717 | DIR=0.9150 | Consistency=0.8660 | Acc=0.7833 | BalAcc=0.6802

Dataset: COMPAS
[Original labels ] DP_Diff=0.1323 | DIR=0.7828 | Consistency=0.6181
Split: train=3694, test=1584
[Baseline LR     ] DP_Diff=0.2553 | DIR=0.6521 | Cons

## 6. Summary table
`Pos_Rate` is the share of favorable predictions; it is shown so that a trivially "fair" classifier can be spotted.

In [11]:
cols = ["Dataset", "Stage", "DP_Diff", "DIR", "Consistency",
        "Accuracy", "Balanced_Acc", "F1_Score", "Pos_Rate"]
summary = results_df[cols].round(4)
display(summary.fillna("-"))
summary.to_csv("lfr_fairness_evaluation_results.csv", index=False)
print("Saved to lfr_fairness_evaluation_results.csv")

,Dataset,Stage,DP_Diff,DIR,Consistency,Accuracy,Balanced_Acc,F1_Score,Pos_Rate
0,Adult,"1. Original labels (full, pre-split)",0.2002,0.3622,0.8327,-,-,-,-
1,Adult,2. Original labels (test split),0.2001,0.3626,0.8278,-,-,-,-
2,Adult,3. Baseline LR (unmitigated),0.2006,0.2756,0.9478,0.8448,0.7676,0.6633,0.2118
3,Adult,4. LFR + LR (mitigated),0.0283,0.8672,0.9598,0.8224,0.7322,0.6078,0.2038
4,German,"1. Original labels (full, pre-split)",0.1494,0.7948,0.7402,-,-,-,-
5,German,2. Original labels (test split),0.1494,0.7948,0.7333,-,-,-,-
6,German,3. Baseline LR (unmitigated),0.1501,0.8238,0.8573,0.77,0.6675,0.849,0.8233
7,German,4. LFR + LR (mitigated),0.0717,0.9150,0.8660,0.7833,0.6802,0.8584,0.83
8,COMPAS,"1. Original labels (full, pre-split)",0.1323,0.7828,0.6181,-,-,-,-
9,COMPAS,2. Original labels (test split),0.1322,0.7828,0.6582,-,-,-,-


Saved to lfr_fairness_evaluation_results.csv


## 7. Comparison of the fairness measures before and after mitigation
Changes are reported relative to (a) the original labels of the full dataset (assignment step 2) and (b) the unmitigated baseline classifier on the same test set. For DP_Diff lower is better; for DIR and Consistency closer to 1 is better (for DIR we also report $|1-\text{DIR}|$).

In [12]:
def get(ds_name, stage_prefix):
    return results_df[(results_df.Dataset == ds_name) &
                      (results_df.Stage.str.startswith(stage_prefix))].iloc[0]

comp = []
for name in datasets:
    orig, base, lfr_row = get(name, "1."), get(name, "3."), get(name, "4.")
    for metric in ["DP_Diff", "DIR", "Consistency"]:
        comp.append({
            "Dataset": name, "Metric": metric,
            "Original labels": orig[metric],
            "Baseline LR": base[metric],
            "LFR + LR": lfr_row[metric],
            "Δ vs original": lfr_row[metric] - orig[metric],
            "Δ vs baseline": lfr_row[metric] - base[metric],
        })
    comp.append({"Dataset": name, "Metric": "|1 - DIR|",
                 "Original labels": abs(1 - orig.DIR), "Baseline LR": abs(1 - base.DIR),
                 "LFR + LR": abs(1 - lfr_row.DIR),
                 "Δ vs original": abs(1 - lfr_row.DIR) - abs(1 - orig.DIR),
                 "Δ vs baseline": abs(1 - lfr_row.DIR) - abs(1 - base.DIR)})
    comp.append({"Dataset": name, "Metric": "Accuracy",
                 "Original labels": np.nan, "Baseline LR": base.Accuracy,
                 "LFR + LR": lfr_row.Accuracy, "Δ vs original": np.nan,
                 "Δ vs baseline": lfr_row.Accuracy - base.Accuracy})

comparison_df = pd.DataFrame(comp).round(4)
display(comparison_df.fillna("-"))
comparison_df.to_csv("lfr_before_after_comparison.csv", index=False)

,Dataset,Metric,Original labels,Baseline LR,LFR + LR,Δ vs original,Δ vs baseline
0,Adult,DP_Diff,0.2002,0.2006,0.0283,-0.1719,-0.1723
1,Adult,DIR,0.3622,0.2756,0.8672,0.505,0.5916
2,Adult,Consistency,0.8327,0.9478,0.9598,0.1271,0.0120
3,Adult,|1 - DIR|,0.6378,0.7244,0.1328,-0.505,-0.5916
4,Adult,Accuracy,-,0.8448,0.8224,-,-0.0224
5,German,DP_Diff,0.1494,0.1501,0.0717,-0.0778,-0.0784
6,German,DIR,0.7948,0.8238,0.9150,0.1202,0.0912
7,German,Consistency,0.7402,0.8573,0.8660,0.1258,0.0087
8,German,|1 - DIR|,0.2052,0.1762,0.0850,-0.1202,-0.0912
9,German,Accuracy,-,0.7700,0.7833,-,0.0133
